# Sovereign Optimizer — GPU benchmark (Kaggle)

Runs the sovereign PDLP solver on an NVIDIA GPU and records results next to the CPU runs.

**Setup (once):**
1. On your laptop run `python tools/make_kaggle_bundle.py` → creates `dist/sovereign_opt_bundle.zip` (code + benchmark data, no node_modules).
2. Kaggle → *Datasets* → *New dataset* → upload the zip, name it `sovereign-opt`.
3. Create a notebook, *Add input* → your dataset, *Settings* → **Accelerator: GPU T4 x2 (or P100)**, *Internet: on*.
4. Upload this notebook (File → Import) and *Run all*.
5. Download the files from `/kaggle/working/results/` and copy them into `benchmarks/results/` in the repo.

In [ ]:
import glob, os, shutil, sys
# Kaggle unpacks uploaded zips automatically, possibly one folder deeper. Find the repo root by its files.
hits = glob.glob('/kaggle/input/**/benchmarks/instances.py', recursive=True)
zips = glob.glob('/kaggle/input/**/sovereign_opt_bundle.zip', recursive=True)
work = '/kaggle/working/sovereign'
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), work, dirs_exist_ok=True)
elif zips:
    shutil.unpack_archive(zips[0], work)
else:
    raise FileNotFoundError('repo not found under /kaggle/input: ' + str(glob.glob('/kaggle/input/*/*')))
os.chdir(work)
os.environ['PYTHONPATH'] = work + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path.insert(0, work)
print(os.getcwd(), sorted(os.listdir('.')))
assert os.path.exists('benchmarks/instances.py') and os.path.exists('sovereign_opt')

In [ ]:
!pip install -q highspy numba
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# 1) GPU sanity check: PDLP on CUDA must agree with CPU on a real Netlib instance
!python -m benchmarks.gpu_parity

In [ ]:
# 2) Scaling curve: 10k .. 3M variables, HiGHS vs PDLP-CPU vs PDLP-GPU
!python -m benchmarks.scale --sizes 1e4 1e5 1e6 3e6 --methods highs pdlp_cpu pdlp_cuda --time-limit 900 --tag kaggle_gpu

In [ ]:
# 3) Netlib with the hybrid GPU pipeline (PDLP on GPU -> simplex crossover -> certificate)
!python -m benchmarks.compare --suite netlib --algorithm hybrid_pdlp --time-limit 300 --tag kaggle_gpu_hybrid

In [ ]:
os.makedirs('/kaggle/working/results', exist_ok=True)
for f in glob.glob('benchmarks/results/*kaggle*'):
    shutil.copy(f, '/kaggle/working/results/')
print(os.listdir('/kaggle/working/results'))